# LU/LC Classification — Classification

This notebook trains four algorithms (AdaBoost, CatBoost, LightGBM, XGBoost)
for **one band configuration**, evaluates them, and saves the prediction maps.
**Run again for each different band set.**

**Generated outputs:**
- `results/models/model_{band}_{algorithm}.pkl` — Trained models
- `results/predictions/pred_{band}_{algorithm}.tif` — Full-image prediction maps
- `results/tables/runlog.csv` — Accuracy metrics (appended/updated)
- `results/logs/` — Timestamped run summary

<br>

> `results/processed/image_9band_masked.tif` and `results/processed/training_labels.tif`
> must already exist (produced by `01_prepare.ipynb`).

## 0 — Configuration

<br>



In [ ]:
import sys
from pathlib import Path

# In Jupyter __file__ is undefined; locate the project root from cwd
_candidates = [p for p in [Path().resolve()] + list(Path().resolve().parents)
               if (p / "config" / "hyperparameters.yaml").exists()]
if not _candidates:
    raise FileNotFoundError(
        "Project root not found. Launch Jupyter from inside the lulc-classification/ directory."
    )
PROJECT_ROOT = _candidates[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================
# BAND_CONFIG — CHANGE ONLY THIS LINE
# Options: "4b", "5b", "6b", "7b", "8b", "9b"
# To add a new band set, add a new block to config/hyperparameters.yaml
# ============================================================
BAND_CONFIG = "9b"

from src.config import PATHS, load_config

cfg        = load_config()
band_cfg   = cfg["band_configs"][BAND_CONFIG]
NUM_BANDS  = band_cfg["num_bands"]
BAND_LABEL = band_cfg["label"]

print(f"Configuration: {BAND_CONFIG} — {BAND_LABEL} ({NUM_BANDS} bands)")
print(f"Band names: {band_cfg['band_names']}")

## 1 — Data Loading

In [ ]:
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

from src.data_loading import load_raster_as_array, load_image_bands

# Load training mask
geo_meta, training_data = load_raster_as_array(PATHS["training_labels"])

# Common NoData mask: derived from all 9 bands
# → any pixel that is NoData in at least one band is excluded from ALL configurations
# → 4b / 5b / 6b / 9b prediction TIFs share the same spatial coverage
_image_9b    = load_image_bands(PATHS["image_masked"], 9)
_nodata_mask = np.isnan(_image_9b).any(axis=-1)   # True → NaN in at least one band

# Use only NUM_BANDS bands for features; apply the common mask
image_data = _image_9b[:, :, :NUM_BANDS].copy()
image_data[_nodata_mask] = np.nan

print(f"Image shape         : {image_data.shape}")
print(f"Training mask shape : {training_data.shape}")
print(f"Common NoData pixels: {int(_nodata_mask.sum())} / {_nodata_mask.size}")

## 2 — Feature Matrix & Labels

In [ ]:
# Keep only labelled pixels
X = image_data[~np.isnan(training_data)]
y = training_data[~np.isnan(training_data)].astype(int)

# Fill NaN values with KNN imputer
imputer = KNNImputer(n_neighbors=1)
X = imputer.fit_transform(X)
y = imputer.fit_transform(y.reshape(-1, 1)).ravel()

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"NaN (X): {np.isnan(X).sum()}, NaN (y): {np.isnan(y).sum()}")

## 3 — Train / Test Split and Class Distribution

The labelled pixel dataset is randomly split into **70% training / 30% test** subsets.

- **Stratified split** (`stratify=y`): Each class's proportion in the training and test
  sets is kept close to the original distribution, eliminating the risk of rare classes
  being absent from the test set.
- **`random_state=42`**: Fixed seed for reproducibility — the same split is obtained
  on every run.
- **Cross-validation** (`cv=5`): Hyperparameter search is evaluated with 5-fold
  stratified CV — independent of this split.

In [ ]:
import pandas as pd
from src.metrics import CLASS_NAMES

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Number of classes: {len(np.unique(y_train))}")

# Class distribution table
all_classes = sorted(set(y_train.astype(int)) | set(y_test.astype(int)))
dist_df = pd.DataFrame({
    "Train": pd.Series(y_train.astype(int)).value_counts().reindex(all_classes, fill_value=0),
    "Test":  pd.Series(y_test.astype(int)).value_counts().reindex(all_classes, fill_value=0),
})
dist_df["Total"] = dist_df.sum(axis=1)
dist_df.loc["TOTAL"] = dist_df.sum()
dist_df.index = [CLASS_NAMES.get(i, str(i)) if i != "TOTAL" else "TOTAL"
                 for i in dist_df.index]

# Percentage column (class share of all pixels)
_grand = dist_df.loc["TOTAL", "Total"]
dist_df["Percent"] = (dist_df["Total"] / _grand * 100).round(2)
dist_df.loc["TOTAL", "Percent"] = 100.00

dist_df


## 4 — AdaBoost

**Adaptive Boosting:** An ensemble method that iteratively combines serial weak
learners (decision trees) by up-weighting misclassified samples.

Hyperparameters searched (`config/hyperparameters.yaml` → `AdaBoost`):
- `estimator__max_depth`: Base decision tree depth
- `n_estimators`: Number of trees in the ensemble
- `learning_rate`: Contribution weight of each learner (smaller → slower but more stable)

In [ ]:
import time
import joblib
from sklearn.metrics import classification_report, cohen_kappa_score
from src.models import build_adaboost, train_classifier
from src.raster_io import save_prediction_tif

PATHS["models"].mkdir(parents=True, exist_ok=True)
model_path_ab = PATHS["models"] / f"model_{BAND_CONFIG}_adaboost.pkl"
param_grid_ab = band_cfg["AdaBoost"]

if model_path_ab.exists():
    saved_ab       = joblib.load(model_path_ab)
    best_adaboost  = saved_ab["model"]
    best_params_ab = saved_ab["best_params"]
    train_time_ab  = None
    print(f"Model loaded : {model_path_ab.name}")
    print(f"Saved parameters: {best_params_ab}")
else:
    t0             = time.time()
    gs_adaboost    = train_classifier(build_adaboost(), param_grid_ab, X_train, y_train)
    train_time_ab  = time.time() - t0
    best_adaboost  = gs_adaboost.best_estimator_
    best_params_ab = gs_adaboost.best_params_
    joblib.dump({"model": best_adaboost, "best_params": best_params_ab}, model_path_ab)
    print(f"Model saved: {model_path_ab.name}")
    print(f"Best parameters: {best_params_ab}")
    print(f"Training time: {train_time_ab:.1f}s")

# Evaluation
y_pred_adaboost        = best_adaboost.predict(X_test)
kappa_adaboost         = cohen_kappa_score(y_test, y_pred_adaboost)
feature_importances_ab = best_adaboost.feature_importances_

report_dict_ab = classification_report(y_test, y_pred_adaboost, output_dict=True, zero_division=0)
weighted_precision_ab = report_dict_ab["weighted avg"]["precision"]
weighted_recall_ab    = report_dict_ab["weighted avg"]["recall"]
weighted_f1_ab        = report_dict_ab["weighted avg"]["f1-score"]

print(classification_report(y_test, y_pred_adaboost, zero_division=0))
print(f"Kappa: {kappa_adaboost:.4f}")
print(f"Feature importances: {feature_importances_ab}")


In [ ]:
# Full-image prediction → save as GeoTIFF
pred_path_ab = PATHS["predictions"] / f"pred_{BAND_CONFIG}_adaboost.tif"
if pred_path_ab.exists():
    print(f"TIF exists, skipping: {pred_path_ab.name}")
    tif_time_ab = None
else:
    t0 = time.time()
    save_prediction_tif(best_adaboost, image_data, geo_meta, pred_path_ab)
    tif_time_ab = time.time() - t0
    print(f"TIF write time: {tif_time_ab:.1f}s")


## 5 — CatBoost

**Categorical Boosting (Yandex):** A gradient boosting implementation that reduces
target leakage using an ordered boosting approach. Generalises well on small sample
sizes and imbalanced classes.

Hyperparameters searched (`config/hyperparameters.yaml` → `CatBoost`):
- `iterations`: Number of trees in the ensemble
- `depth`: Tree depth
- `learning_rate`: Gradient step size

In [ ]:
from src.models import build_catboost

model_path_cb = PATHS["models"] / f"model_{BAND_CONFIG}_catboost.pkl"
param_grid_cb = band_cfg["CatBoost"]

if model_path_cb.exists():
    saved_cb       = joblib.load(model_path_cb)
    best_catboost  = saved_cb["model"]
    best_params_cb = saved_cb["best_params"]
    train_time_cb  = None
    print(f"Model loaded : {model_path_cb.name}")
    print(f"Saved parameters: {best_params_cb}")
else:
    t0             = time.time()
    gs_catboost    = train_classifier(build_catboost(), param_grid_cb, X_train, y_train)
    train_time_cb  = time.time() - t0
    best_catboost  = gs_catboost.best_estimator_
    best_params_cb = gs_catboost.best_params_
    joblib.dump({"model": best_catboost, "best_params": best_params_cb}, model_path_cb)
    print(f"Model saved: {model_path_cb.name}")
    print(f"Best parameters: {best_params_cb}")
    print(f"Training time: {train_time_cb:.1f}s")

# Evaluation
y_pred_catboost        = best_catboost.predict(X_test)
kappa_catboost         = cohen_kappa_score(y_test, y_pred_catboost)
feature_importances_cb = best_catboost.feature_importances_

report_dict_cb = classification_report(y_test, y_pred_catboost, output_dict=True, zero_division=0)
weighted_precision_cb = report_dict_cb["weighted avg"]["precision"]
weighted_recall_cb    = report_dict_cb["weighted avg"]["recall"]
weighted_f1_cb        = report_dict_cb["weighted avg"]["f1-score"]

print(classification_report(y_test, y_pred_catboost, zero_division=0))
print(f"Kappa: {kappa_catboost:.4f}")


In [ ]:
pred_path_cb = PATHS["predictions"] / f"pred_{BAND_CONFIG}_catboost.tif"
if pred_path_cb.exists():
    print(f"TIF exists, skipping: {pred_path_cb.name}")
    tif_time_cb = None
else:
    t0 = time.time()
    save_prediction_tif(best_catboost, image_data, geo_meta, pred_path_cb)
    tif_time_cb = time.time() - t0
    print(f"TIF write time: {tif_time_cb:.1f}s")


## 6 — LightGBM

**Light Gradient Boosting Machine (Microsoft):** Histogram-based leaf-wise growth
strategy offering memory efficiency and speed on large datasets. Captures more complex
patterns than level-wise growth, but requires careful regularisation to control
overfitting.

Hyperparameters searched (`config/hyperparameters.yaml` → `LightGBM`):
- `n_estimators`: Number of trees in the ensemble
- `max_depth`: Maximum tree depth (−1 = unlimited)
- `learning_rate`: Gradient step size

In [ ]:
from src.models import build_lightgbm
import lightgbm as lgb

model_path_lgb = PATHS["models"] / f"model_{BAND_CONFIG}_lightgbm.pkl"
param_grid_lgb = band_cfg["LightGBM"]

if model_path_lgb.exists():
    saved_lgb       = joblib.load(model_path_lgb)
    best_lightgbm   = saved_lgb["model"]
    best_params_lgb = saved_lgb["best_params"]
    train_time_lgb  = None
    print(f"Model loaded : {model_path_lgb.name}")
    print(f"Saved parameters: {best_params_lgb}")
else:
    t0 = time.time()
    X_train_lgb = pd.DataFrame(X_train, columns=band_cfg["band_names"])
    gs_lightgbm = train_classifier(
        build_lightgbm(), param_grid_lgb, X_train_lgb, y_train,
        fit_params={"callbacks": [lgb.log_evaluation(-1)]},
    )
    train_time_lgb  = time.time() - t0
    best_lightgbm   = gs_lightgbm.best_estimator_
    best_params_lgb = gs_lightgbm.best_params_
    joblib.dump({"model": best_lightgbm, "best_params": best_params_lgb}, model_path_lgb)
    print(f"Model saved: {model_path_lgb.name}")
    print(f"Best parameters: {best_params_lgb}")
    print(f"Training time: {train_time_lgb:.1f}s")

# Evaluation
X_test_lgb              = pd.DataFrame(X_test, columns=band_cfg["band_names"])
y_pred_lightgbm         = best_lightgbm.predict(X_test_lgb)
kappa_lightgbm          = cohen_kappa_score(y_test, y_pred_lightgbm)
feature_importances_lgb = best_lightgbm.feature_importances_

report_dict_lgb = classification_report(y_test, y_pred_lightgbm, output_dict=True, zero_division=0)
weighted_precision_lgb = report_dict_lgb["weighted avg"]["precision"]
weighted_recall_lgb    = report_dict_lgb["weighted avg"]["recall"]
weighted_f1_lgb        = report_dict_lgb["weighted avg"]["f1-score"]

print(classification_report(y_test, y_pred_lightgbm, zero_division=0))
print(f"Kappa: {kappa_lightgbm:.4f}")


In [ ]:
pred_path_lgb = PATHS["predictions"] / f"pred_{BAND_CONFIG}_lightgbm.tif"
if pred_path_lgb.exists():
    print(f"TIF exists, skipping: {pred_path_lgb.name}")
    tif_time_lgb = None
else:
    best_lightgbm.set_params(n_jobs=-1)
    t0 = time.time()
    save_prediction_tif(best_lightgbm, image_data, geo_meta, pred_path_lgb)
    tif_time_lgb = time.time() - t0
    print(f"TIF write time: {tif_time_lgb:.1f}s")


## 7 — XGBoost

**Extreme Gradient Boosting (DMLC):** A gradient boosting implementation with
depth-wise growth, L1/L2 regularisation, and second-order (Newton-Raphson)
optimisation. Widely cited in the comparative literature and frequently used as a
baseline.

Hyperparameters searched (`config/hyperparameters.yaml` → `XGBoost`):
- `n_estimators`: Number of trees in the ensemble
- `max_depth`: Tree depth
- `learning_rate`: Gradient step size (eta)

In [ ]:
from src.models import build_xgboost

model_path_xgb = PATHS["models"] / f"model_{BAND_CONFIG}_xgboost.pkl"
param_grid_xgb = band_cfg["XGBoost"]

if model_path_xgb.exists():
    saved_xgb       = joblib.load(model_path_xgb)
    best_xgboost    = saved_xgb["model"]
    best_params_xgb = saved_xgb["best_params"]
    train_time_xgb  = None
    print(f"Model loaded : {model_path_xgb.name}")
    print(f"Saved parameters: {best_params_xgb}")
else:
    t0              = time.time()
    gs_xgboost      = train_classifier(build_xgboost(), param_grid_xgb, X_train, y_train)
    train_time_xgb  = time.time() - t0
    best_xgboost    = gs_xgboost.best_estimator_
    best_params_xgb = gs_xgboost.best_params_
    joblib.dump({"model": best_xgboost, "best_params": best_params_xgb}, model_path_xgb)
    print(f"Model saved: {model_path_xgb.name}")
    print(f"Best parameters: {best_params_xgb}")
    print(f"Training time: {train_time_xgb:.1f}s")

# Evaluation
y_pred_xgboost          = best_xgboost.predict(X_test)
kappa_xgboost           = cohen_kappa_score(y_test, y_pred_xgboost)
feature_importances_xgb = best_xgboost.feature_importances_

report_dict_xgb = classification_report(y_test, y_pred_xgboost, output_dict=True, zero_division=0)
weighted_precision_xgb = report_dict_xgb["weighted avg"]["precision"]
weighted_recall_xgb    = report_dict_xgb["weighted avg"]["recall"]
weighted_f1_xgb        = report_dict_xgb["weighted avg"]["f1-score"]

print(classification_report(y_test, y_pred_xgboost, zero_division=0))
print(f"Kappa: {kappa_xgboost:.4f}")

In [ ]:
pred_path_xgb = PATHS["predictions"] / f"pred_{BAND_CONFIG}_xgboost.tif"
if pred_path_xgb.exists():
    print(f"TIF exists, skipping: {pred_path_xgb.name}")
    tif_time_xgb = None
else:
    t0 = time.time()
    save_prediction_tif(best_xgboost, image_data, geo_meta, pred_path_xgb)
    tif_time_xgb = time.time() - t0
    print(f"TIF write time: {tif_time_xgb:.1f}s")


## 8 — Runlog Update

Results for all four algorithms are appended to `results/runlog.csv`.
If the same `(Algorithm, Bands)` combination exists, it is overwritten.

In [ ]:
from src.metrics import append_band_to_log

results = {
    "AdaBoost": {"y_pred": y_pred_adaboost, "kappa": kappa_adaboost, "fi": feature_importances_ab},
    "CatBoost": {"y_pred": y_pred_catboost, "kappa": kappa_catboost, "fi": feature_importances_cb},
    "LightGBM": {"y_pred": y_pred_lightgbm, "kappa": kappa_lightgbm, "fi": feature_importances_lgb},
    "XGBoost":  {"y_pred": y_pred_xgboost,  "kappa": kappa_xgboost,  "fi": feature_importances_xgb},
}

append_band_to_log(BAND_LABEL, y_test, results)

## 9 — Run Log

A timestamped summary file is saved to `results/logs/` on every run.

In [ ]:
import datetime
# File name: {raw_image}_{YYYY_MM_DD}_{HH_MM_SS}.txt
_now      = datetime.datetime.now()
_img_stem = PATHS["raw_image"].name
_log_out  = PATHS["run_logs"]                  # results/logs/
_log_out.mkdir(parents=True, exist_ok=True)
_log_path = _log_out / f"{_img_stem}_{_now.strftime('%Y_%m_%d_%H_%M_%S')}.txt"

# Band statistics (non-NaN pixels)
_band_names = band_cfg["band_names"]
_band_stats = []
for _i in range(NUM_BANDS):
    _b     = _image_9b[:, :, _i]
    _valid = _b[~np.isnan(_b)]
    _band_stats.append({
        "name":  _band_names[_i],
        "min":   float(_valid.min()),
        "max":   float(_valid.max()),
        "mean":  float(_valid.mean()),
        "std":   float(_valid.std()),
    })

# Image size, study area, and pixel resolution
_h, _w      = image_data.shape[:2]
_px         = abs(geo_meta[0][1])                          # GeoTransform[1] = pixel width (metres)
_valid_px   = int((~np.isnan(_image_9b[:, :, 0])).sum())   # valid pixels after study_area mask
_area_ha    = _valid_px * (_px ** 2) / 10_000              # pixel area → hectares

# Log content
_lines = [
    "=== Run Log ===",
    f"Date           : {_now.strftime('%Y-%m-%d %H:%M:%S')}",
    f"BAND_CONFIG    : {BAND_CONFIG} ({BAND_LABEL})",
    f"Band names     : {_band_names}",
    f"Full image     : {_w} × {_h} pixels",
    f"Study area     : {_valid_px:,} pixels  ({_area_ha:.1f} ha)",
    f"Pixel size     : {_px:.2f} m",
    "",
    "--- Band Statistics (valid pixels) ---",
]
for _s in _band_stats:
    _lines.append(
        f"  {_s['name']:12s}: min={_s['min']:.4g}  max={_s['max']:.4g}"
        f"  mean={_s['mean']:.4g}  std={_s['std']:.4g}"
    )

_lines += ["", "--- Hyperparameter Optimisation Results ---"]
for _algo, _pg, _bp in [
    ("AdaBoost",  param_grid_ab,  best_params_ab),
    ("CatBoost",  param_grid_cb,  best_params_cb),
    ("LightGBM",  param_grid_lgb, best_params_lgb),
    ("XGBoost",   param_grid_xgb, best_params_xgb),
]:
    _kw = max(len(k) for k in _pg)
    _lines.append(f"\n{_algo}:")
    _lines.append("  Search space:")
    for _k, _v in _pg.items():
        _vals_str = ", ".join(str(x) for x in _v)
        _lines.append(f"    {_k:{_kw}s} : {_vals_str}")
    _lines.append("  Selected parameters:")
    for _k, _v in _bp.items():
        _lines.append(f"    {_k:{_kw}s} : {_v}")

_lines += ["", "--- Timing ---"]
for _algo, _tr, _tf in [
    ("AdaBoost",  train_time_ab,  tif_time_ab),
    ("CatBoost",  train_time_cb,  tif_time_cb),
    ("LightGBM",  train_time_lgb, tif_time_lgb),
    ("XGBoost",   train_time_xgb, tif_time_xgb),
]:
    _tr_str = f"{_tr:.1f}s" if _tr is not None else "loaded from disk"
    _tf_str = f"{_tf:.1f}s" if _tf is not None else "existing TIF used"
    _lines.append(f"  {_algo:10s}: train={_tr_str}  TIF={_tf_str}")

_log_text = "\n".join(_lines)
_log_path.write_text(_log_text, encoding="utf-8")
print(f"Log saved: {_log_path}")
print(_log_text)